# Step 2a: Stratified Sample Selection

Select 100 representative job ads from the 50,000 ads dataset for annotation and model evaluation.

## Sampling Strategy

We use **stratified random sampling** based on job classifications to ensure the sample reflects the diversity of the full dataset.

### Justification

**Why 100 ads?**
- Large enough to capture diversity across job types and industries
- Small enough to enable thorough validation

**Why stratified sampling?**
- **Representativeness**: Ensures all major job classifications are included proportionally (Does not focus on specific industries, reducing risk of missing small but important job categories)
- **Reduces bias**: Prevents over-representation of dominant categories (e.g., ICT)
- **Industry diversity**: Different industries use different vocabulary for similar skills
- **Better generalization**: Model trained/evaluated on diverse sample will perform better on unseen data

In [ ]:
import json
import pandas as pd
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42) 

## Load Data

In [ ]:
# Load all job ads
ads = []
with open('ads-50k.json', 'r', encoding='utf-8') as f:
    for line in f:
        ads.append(json.loads(line))

print(f"Total ads loaded: {len(ads):,}")

## Analyze Classification Distribution

In [ ]:
# Extract classifications
classifications = [ad['metadata'].get('classification', {}).get('name', 'Unknown') for ad in ads]
classification_counts = Counter(classifications)

# Create distribution DataFrame
dist_df = pd.DataFrame(
    classification_counts.most_common(),
    columns=['Classification', 'Count']
)
dist_df['Percentage'] = (dist_df['Count'] / len(ads) * 100).round(2)

print("Full Dataset Distribution:")
print(dist_df.head(20))

## Stratified Sampling

In [ ]:
# Organize ads by classification
ads_by_classification = {}
for ad in ads:
    cls = ad['metadata'].get('classification', {}).get('name', 'Unknown')
    if cls not in ads_by_classification:
        ads_by_classification[cls] = []
    ads_by_classification[cls].append(ad)

# Calculate sample size per classification (proportional)
sample_size = 100
sampling_plan = {}

for cls, count in classification_counts.most_common():
    proportion = count / len(ads)
    n_samples = int(np.round(sample_size * proportion))
    # Ensure at least 1 sample for each classification that exists
    if n_samples == 0 and count > 0:
        n_samples = 1
    sampling_plan[cls] = n_samples

# Adjust to exactly 100 (due to rounding)
total_planned = sum(sampling_plan.values())
if total_planned != sample_size:
    # Adjust the largest category
    largest_cls = classification_counts.most_common(1)[0][0]
    sampling_plan[largest_cls] += (sample_size - total_planned)

print("\nSampling Plan:")
plan_df = pd.DataFrame([
    {'Classification': cls, 'Full Dataset Count': classification_counts[cls], 
     'Sample Count': sampling_plan[cls]}
    for cls in sampling_plan.keys()
]).sort_values('Full Dataset Count', ascending=False)

print(plan_df.head(20))
print(f"\nTotal sample size: {plan_df['Sample Count'].sum()}")

In [ ]:
# Perform stratified sampling
sampled_ads = []

for cls, n_samples in sampling_plan.items():
    if n_samples > 0:
        available_ads = ads_by_classification[cls]
        # Sample without replacement
        n_to_sample = min(n_samples, len(available_ads))
        sampled = np.random.choice(available_ads, size=n_to_sample, replace=False)
        sampled_ads.extend(sampled)

print(f"Total ads sampled: {len(sampled_ads)}")

## Validate Sample

In [ ]:
# Compare sample distribution to full dataset
sample_classifications = [ad['metadata'].get('classification', {}).get('name', 'Unknown') 
                          for ad in sampled_ads]
sample_counts = Counter(sample_classifications)

comparison_df = pd.DataFrame({
    'Classification': list(classification_counts.keys()),
    'Full Dataset %': [classification_counts[cls] / len(ads) * 100 
                       for cls in classification_counts.keys()],
    'Sample %': [sample_counts[cls] / len(sampled_ads) * 100 
                 for cls in classification_counts.keys()]
}).sort_values('Full Dataset %', ascending=False)

comparison_df['Difference'] = (comparison_df['Sample %'] - comparison_df['Full Dataset %']).round(2)

print("\nDistribution Comparison (Top 15):")
print(comparison_df.head(15).to_string(index=False))

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(12, 8))

top_n = 15
top_classifications = comparison_df.head(top_n)

x = np.arange(len(top_classifications))
width = 0.35

ax.barh(x - width/2, top_classifications['Full Dataset %'], width, label='Full Dataset', alpha=0.8)
ax.barh(x + width/2, top_classifications['Sample %'], width, label='Sample (n=100)', alpha=0.8)

ax.set_yticks(x)
ax.set_yticklabels(top_classifications['Classification'], fontsize=9)
ax.set_xlabel('Percentage (%)')
ax.set_title(f'Distribution Comparison: Top {top_n} Job Classifications')
ax.legend()
ax.invert_yaxis()

plt.tight_layout()
plt.show()

print(f"\nMean absolute difference: {comparison_df['Difference'].abs().mean():.2f}%")
print(f"Max absolute difference: {comparison_df['Difference'].abs().max():.2f}%")

## Save Sample

In [ ]:
sample_data = []

for ad in sampled_ads:
    sample_data.append({
        'id': ad['id'],
        'title': ad['title'],
        'abstract': ad['abstract'],
        'content': ad['content'],
        'metadata': json.dumps(ad['metadata'])  # Store metadata as JSON string
    })

sample_df = pd.DataFrame(sample_data)

# Save to CSV
output_file = 'sampled_100_ads.csv'
sample_df.to_csv(output_file, index=False, encoding='utf-8')

print(f"Sample saved to: {output_file}")
print(f"Shape: {sample_df.shape}")
print(f"\nColumns: {list(sample_df.columns)}")

In [ ]:
# Display first few rows
print("\nFirst 3 sampled ads:")
for idx in range(min(3, len(sample_df))):
    row = sample_df.iloc[idx]
    print(f"\n{idx+1}. ID: {row['id']}")
    print(f"   Title: {row['title']}")
    print(f"   Classification: {json.loads(row['metadata']).get('classification', {}).get('name', 'N/A')}")
    print(f"   Abstract: {row['abstract'][:100]}...")

## Summary
- 100 job ads selected via stratified random sampling
- Proportional representation of all major job classifications